# SVM Analytics Drill-Down

This notebook auto-loads the latest analytics run from `output/analytics/` and provides interactive drill-down views for:

- threshold tradeoffs
- review-volume behavior
- top feature weights
- generated PNG charts

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'output').exists():
    BASE_DIR = BASE_DIR.parent

ANALYTICS_ROOT = BASE_DIR / 'output' / 'analytics'
if not ANALYTICS_ROOT.exists():
    raise FileNotFoundError(f'Analytics folder not found: {ANALYTICS_ROOT}')

run_dirs = [d for d in ANALYTICS_ROOT.iterdir() if d.is_dir() and d.name.startswith('svm_analytics_')]
if not run_dirs:
    raise FileNotFoundError('No analytics run folders found under output/analytics. Run: python scripts/svm_analytics.py')

latest_run = max(run_dirs, key=lambda p: p.stat().st_mtime)
print(f'Latest analytics run: {latest_run}')

In [ ]:
threshold_csv = latest_run / 'threshold_sweep.csv'
features_csv = latest_run / 'top_features.csv'

threshold_df = pd.read_csv(threshold_csv)
features_df = pd.read_csv(features_csv)

print('Threshold sweep rows:', len(threshold_df))
print('Top features rows:', len(features_df))

display(threshold_df.head(10))
display(features_df.head(20))

In [ ]:
# Drill-down: choose a threshold and inspect expected behavior
selected_threshold = float(threshold_df.loc[threshold_df['f1_1'].idxmax(), 'threshold'])
nearest_idx = (threshold_df['threshold'] - selected_threshold).abs().idxmin()
row = threshold_df.loc[nearest_idx]

print(f'Selected threshold: {row["threshold"]:.4f}')
print(f'Accuracy: {row["accuracy"]:.4f}')
print(f'Class-1 Precision: {row["precision_1"]:.4f}')
print(f'Class-1 Recall: {row["recall_1"]:.4f}')
print(f'Class-1 F1: {row["f1_1"]:.4f}')
print(f'Estimated review rate: {row["review_rate"]:.2%}')

In [ ]:
# Plot threshold tradeoff interactively in notebook
plt.figure(figsize=(10, 5))
plt.plot(threshold_df['threshold'], threshold_df['precision_1'], label='precision_1')
plt.plot(threshold_df['threshold'], threshold_df['recall_1'], label='recall_1')
plt.plot(threshold_df['threshold'], threshold_df['f1_1'], label='f1_1')
plt.axvline(selected_threshold, color='red', linestyle='--', label='selected threshold')
plt.ylim(0, 1.05)
plt.xlabel('Decision Threshold')
plt.ylabel('Metric')
plt.title('Class-1 Threshold Tradeoff (Notebook Drill-Down)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Show generated analytics images from latest run
image_files = [
    'confusion_matrix.png',
    'class_metrics.png',
    'threshold_tradeoff.png',
    'confidence_distribution.png',
    'review_volume.png',
    'top_features.png',
]

for img_name in image_files:
    img_path = latest_run / img_name
    if not img_path.exists():
        print(f'Missing image: {img_path}')
        continue

    img = mpimg.imread(img_path)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.title(img_name)
    plt.axis('off')
    plt.tight_layout()
    plt.show()